In [1]:
!python --version

import torch
print(torch.__version__)
print(torch.cuda.is_available())

!pip install -U transformers
!pip install -U datasets
!pip install tensorboard
!pip install sentencepiece
!pip install accelerate
!pip install evaluate==0.4.0
!pip install rouge_score
!pip install bleu
!pip install -U scikit-learn
!pip install nltk datasets
!pip install sacrebleu
!pip install nltk 

Python 3.10.15
2.5.1+cu118
True
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [2]:
import os
import torch
import sacrebleu
import numpy as np
import nltk
import evaluate

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

os.environ["CUDA_VISIBLE_DEVICES"]="1,3"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:32"

torch.cuda.empty_cache()


In [3]:
import json
lista_pictogramas_ids=[]
with open("ids_arasaac.txt", 'r', encoding='utf-8') as archivo:
  lineas = archivo.readlines()
  for linea in lineas:
    lista_pictogramas_ids.append(linea.strip())

print(len(lista_pictogramas_ids))

13369


In [4]:
dataset_train = load_dataset('json', data_files='train_data.json')['train']
dataset_test = load_dataset('json', data_files='test_data.json')['train']
dataset_valid = load_dataset('json', data_files='validation_data.json')['train']

# Mostrar estadísticas básicas
print(dataset_train)
print(dataset_test)
print(dataset_valid)

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['id', 'oracion', 'traduccion'],
    num_rows: 15362
})
Dataset({
    features: ['id', 'oracion', 'traduccion'],
    num_rows: 423
})
Dataset({
    features: ['id', 'oracion', 'traduccion'],
    num_rows: 422
})


In [5]:
model_checkpoint = "flax-community/spanish-t5-small" #"vgaraujov/t5-base-spanish"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

total_params = sum(p.numel() for p in model.parameters())
print(f"{total_params:,} total parameters.")

total_trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad)
print(f"{total_trainable_params:,} training parameters.")

# Agregar nuevos tokens al tokenizer
existing_tokens = set(tokenizer.get_vocab().keys())
new_tokens = [token for token in lista_pictogramas_ids if token not in existing_tokens]
tokenizer.add_tokens(new_tokens)

model.resize_token_embeddings(len(tokenizer))

# Validar parámetros actualizados
print("After adding new tokens:")
total_params = sum(p.numel() for p in model.parameters())
print(f"{total_params:,} total parameters.")
total_trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{total_trainable_params:,} training parameters.")

new_token_ids = tokenizer.convert_tokens_to_ids(new_tokens)
print(f"Sample IDs of new tokens: {new_token_ids[:10]}")

new_embeddings = model.get_input_embeddings().weight.data[new_token_ids]
print("Sample embeddings for new tokens:", new_embeddings[:5])

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Convertir los nuevos tokens a sus IDs
new_token_ids = tokenizer.convert_tokens_to_ids(new_tokens)

# Mostrar ejemplos de tokens agregados y sus IDs
for token, token_id in zip(new_tokens[:10], new_token_ids[:10]):
    print(f"Token: {token}, ID: {token_id}")

60,493,824 total parameters.
60,493,824 training parameters.


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


After adding new tokens:
67,338,752 total parameters.
67,338,752 training parameters.
Sample IDs of new tokens: [32103, 32104, 32105, 32106, 32107, 32108, 32109, 32110, 32111, 32112]
Sample embeddings for new tokens: tensor([[ 0.4681, -0.5677, -0.4633,  ...,  0.7039, -0.5305, -0.0855],
        [ 0.4681, -0.5677, -0.4633,  ...,  0.7039, -0.5305, -0.0855],
        [ 0.4681, -0.5677, -0.4633,  ...,  0.7039, -0.5305, -0.0855],
        [ 0.4681, -0.5677, -0.4633,  ...,  0.7039, -0.5305, -0.0855],
        [ 0.4681, -0.5677, -0.4633,  ...,  0.7039, -0.5305, -0.0855]])
Token: pict_2247, ID: 32103
Token: pict_2248, ID: 32104
Token: pict_2249, ID: 32105
Token: pict_2250, ID: 32106
Token: pict_2251, ID: 32107
Token: pict_2252, ID: 32108
Token: pict_2253, ID: 32109
Token: pict_2254, ID: 32110
Token: pict_2255, ID: 32111
Token: pict_2256, ID: 32112


In [6]:
model.to(torch.device('cuda'))
model.device

device(type='cuda', index=0)

In [7]:
max_input_length = 20
max_target_length = 20

batch_size=16
metric ="bleu"
model_name = "t5-xgen-finetuned"
evaluation_strategy = "epoch"
save_strategy="epoch"
overwrite_output_dir=True
learning_rate=10e-5
gradient_accumulation_steps=1
weight_decay=0.01
do_train=True
do_eval=True
save_total_limit=20
num_train_epochs=20
seed=42
predict_with_generate=True
fp16=True
metric_for_best_model="bleu"
load_best_model_at_end=True
generation_max_length = max_target_length
logging_strategy="epoch"
eval_accumulation_steps=2

In [8]:
def preprocess_function(examples):
    inputs = ["translate: " + oracion for oracion in examples['oracion']]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True, padding="max_length")    
    labels = tokenizer(text_target=examples["traduccion"], max_length=max_target_length, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [9]:
encoded_train = dataset_train.map(preprocess_function, batched=True)
encoded_test = dataset_test.map(preprocess_function, batched=True)
encoded_validation = dataset_valid.map(preprocess_function, batched=True)

Map:   0%|          | 0/15362 [00:00<?, ? examples/s]

Map:   0%|          | 0/423 [00:00<?, ? examples/s]

Map:   0%|          | 0/422 [00:00<?, ? examples/s]

In [10]:
nltk.download("punkt", quiet=True)

bleu_metric = evaluate.load("bleu")
chrf_metric = evaluate.load('chrf')

def compute_metrics_v2(eval_preds):
    print(f"metrics v2")
    predictions, labels = eval_preds

    # Reemplazar -100 con pad_token_id tanto en predicciones como en etiquetas
    pad_token_id = tokenizer.pad_token_id
    predictions = np.where(predictions != -100, predictions, pad_token_id)
    labels = np.where(labels != -100, labels, pad_token_id)
    
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
     
    # Asegurarse de que BLEU reciba cadenas completas, no listas de palabras
    preds = [" ".join(pred.split()) for pred in decoded_preds]
    refs = [[" ".join(label.split())] for label in decoded_labels]  # Lista de listas para referencias
    
    print(f"bleu_preds {preds}")
    print(f"bleu_refs {refs}")
    
    # Calcular BLEU
    try:
        bleu_result = bleu_metric.compute(predictions=preds, references=refs)
        bleu_score = {"bleu": bleu_result["bleu"], "precisions": bleu_result["precisions"]}
    except Exception as e:
        print(f"Error al calcular BLEU: {e}")
        bleu_score = {"bleu": 0.0}

    # Calcular CHRF++
    try:
        chrf_result = chrf_metric.compute(predictions=preds, references=refs)
        print(chrf_result)
        chrf_score = {"chrf": chrf_result["score"]}
    except Exception as e:
        print(f"Error al calcular CHRF++: {e}")
        chrf_score = {"chrf": 0.0}
        
    # Combinar resultados
    combined_results = {**bleu_score, **chrf_score}
    return combined_results

In [11]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
args = Seq2SeqTrainingArguments(
    output_dir="./results",
    evaluation_strategy = evaluation_strategy,
    save_strategy = save_strategy,
    overwrite_output_dir = overwrite_output_dir,
    learning_rate = learning_rate,
    per_device_train_batch_size = batch_size,
    per_device_eval_batch_size= batch_size,
    gradient_accumulation_steps= gradient_accumulation_steps,
    weight_decay= weight_decay,
    do_train= do_train,
    do_eval= do_eval,
    save_total_limit= save_total_limit,
    num_train_epochs= num_train_epochs,
    seed= seed,
    predict_with_generate= predict_with_generate,
    fp16= fp16,
    generation_max_length=generation_max_length,
    logging_strategy=logging_strategy,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    )

trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=encoded_train,
    eval_dataset=encoded_test,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics_v2
)



C:\ProgramData\anaconda3\envs\thesis\lib\site-packages\transformers\training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\apare\AppData\Local\Temp\ipykernel_2084\3537574832.py:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [12]:
import numpy as np

import nltk
nltk.download('punkt')

trainer.train()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\apare\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss,Bleu,Precisions,Chrf
1,2.207200,2.205326,0.000000,"[0.728, 0.23684210526315788, 0.07142857142857142, 0.0]",3.456034
2,1.366800,2.029052,0.253224,"[0.6496794344895611, 0.3899293286219081, 0.17413694449742514, 0.0932062966031483]",44.297943
3,1.050700,2.034837,0.259330,"[0.6602297319793574, 0.39881805157593125, 0.1787579802669762, 0.09608915054667788]",44.939643
4,0.943100,2.043281,0.271016,"[0.6654318908907223, 0.40554749818709207, 0.18639749117992943, 0.1072494669509595]",47.492569
5,0.868600,2.058825,0.282184,"[0.6674453529117303, 0.41526032315978456, 0.20007754943776657, 0.11433986102337335]",48.835026
6,0.804000,2.034623,0.295957,"[0.673992673992674, 0.4264732222819273, 0.21250241919876137, 0.12560488112770882]",50.157383
7,0.744600,2.080672,0.325897,"[0.678030303030303, 0.4383076650734643, 0.23551902121965207, 0.16116303219106956]",51.155281
8,0.689100,2.064388,0.343091,"[0.6900750625521268, 0.4547738693467337, 0.2520372526193248, 0.17517940059096665]",52.143661
9,0.634900,2.070961,0.364908,"[0.6965747702589807, 0.4667385832434376, 0.272108843537415, 0.20042283298097252]",53.409116
10,0.586000,2.074725,0.395581,"[0.7084240495729358, 0.49008651766402306, 0.30416991426344503, 0.23187791437049599]",55.262784


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

metrics v2
bleu_preds ['', 'yo borro', '', '', '', 'proteger', '', '', '', '', 'siembra _crear', '', '', 'explicar', '', '', 'observar', '', '', '', '', '', 'pobreza _en', '', 'turismo rural', '', 'cáscara', '', '', 'responder', '', '', '', '', '', '', '', '', 'responder', 'relatora _que escribas', 'ingredientes', '', '', '', '', '', 'objetos', '', '', '', '', 'prohibido', '', '', '', '', '', '', '', '', 'intercambia', 'prohibido', 'relaciona', '', '', '', '', '', '', '', '', '', '', '', '', 'juntar', '', 'electricidad eólica', '', '', '', '', '', 'reirme', '', '', '', 'extinción _de', '', '', '', 'derechos', '', '', '', '', '', '', '', '', 'inflamación', '', '', 'relacionamos', '', '', '', '', '', '', '', '', '', '', '', 'cita previa', '', '', 'recorta', '', '', 'sentir', '', '', '', '', '', '', '', '', 'adiós', '', '', '', '', 'mina', '', '', '', '', '', 'noticia informativa', 'destinatario _del email', 'ausencia', '', '', 'rebajas', '', '', '', '', '', '', 'sembrar', '', '', '', '',

Trainer is attempting to log a value of "[0.6496794344895611, 0.3899293286219081, 0.17413694449742514, 0.0932062966031483]" of type <class 'list'> for key "eval/precisions" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.


...

metrics v2
bleu_preds ['pict_6901 _ luna pict_11691 _llena', 'pict_3047 _yo borro pict_7029 _la pict_9837 _h', 'pict_3414 _¡ pict_12281 _nuestro pict_6901 _visita pict_3418 _', 'pict_3415 _¿ pict_22619 _dónde pict_5466 _estar pict_3418 _', 'pict_11749 _hacer pict_8474 _un pict_6901 _cartel pict_7074', 'pict_11691 _proteger pict_8476 _el pict_6901 _medio ambiente', 'pict_12281 _tu pict_9819 _turno pict_7074 _de pict_2380', 'pict_6901 _trabajadores pict_7034 _en pict_37340 _cerámica', 'pict_2474 _observar pict_7029 _la pict_9819 _tierra', 'pict_3415 _¿ pict_22619 _dónde pict_2380 _escribir pict_3418 _?', 'pict_6901 _siembra pict_3047 _y pict_9870 _cultiva', 'pict_2474 _observar pict_5466 _estar pict_4604 _sala', 'pict_2474 _escuchar pict_7029 _la pict_9870 _exposición', 'pict_8579 _explicar pict_8474 _una idea', 'pict_3415 _¿ pict_7212 _por pict_22624 _qué pict_3418 _?', 'pict_2474 _observar pict_8476 _el glóbulo pict_4604', 'pict_2474 _revisar pict_8476 _el pict_22769 _martillo', 'pict_

Trainer is attempting to log a value of "[0.7161129568106313, 0.5067000178667143, 0.3298591005597375, 0.26138032305433184]" of type <class 'list'> for key "eval/precisions" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.


metrics v2
bleu_preds ['pict_6901 _ luna pict_11691 _llena', 'pict_3047 _yo borro pict_7029 _la pict_9837 _h', 'pict_3414 _¡ pict_12281 _nuestro pict_6901 _visita pict_3418 _', 'pict_3415 _¿ pict_22619 _dónde pict_5466 _estar pict_3418 _', 'pict_11749 _hacer pict_8474 _un pict_9819 _cartel pict_7074', 'pict_11691 _proteger pict_8476 _el pict_6901 _medio ambiente', 'pict_12281 _tu pict_9819 _turno pict_7074 _de pict_2380', 'pict_6901 _trabajadores pict_7034 _en pict_37340 _cerámica', 'pict_2474 _observar pict_7029 _la pict_9819 _tierra', 'pict_3415 _¿ pict_22619 _dónde pict_2380 _escribir pict_3418 _?', 'pict_6901 _siembra pict_3047 _y pict_9870 _cultiva', 'pict_2474 _observar pict_5466 _estar pict_4604 _sala', 'pict_2474 _escuchar pict_7029 _la pict_9870 _exposición', 'pict_8579 _explicar pict_8474 _una idea', 'pict_3415 _¿ pict_7212 _por pict_22624 _qué pict_3418 _?', 'pict_2474 _observar pict_8476 _el glóbulo pict_4604', 'pict_2474 _revisar pict_8476 _el pict_22769 _martillo', 'pict_

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=19220, training_loss=0.7187406160828971, metrics={'train_runtime': 10457.3614, 'train_samples_per_second': 29.38, 'train_steps_per_second': 1.838, 'total_flos': 1624313089228800.0, 'train_loss': 0.7187406160828971, 'epoch': 20.0})

In [29]:
## Se necesita saber con exactitud como se llama el folder

tokenizerv1 = AutoTokenizer.from_pretrained('./results/checkpoint-2196')
modelv1 = AutoModelForSeq2SeqLM.from_pretrained('./results/checkpoint-2196')

device = torch.device("cuda")
modelv1.to(device)
modelv1.eval()

T5ForConditionalGeneration(
  (shared): Embedding(70918, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(70918, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [32]:
# Función para preparar los datos para la predicción
def prepare_data_for_prediction(dataset, tokenizer):
    # Transformar los datos para que sean aptos para el modelo
    prepared_data = []
    for example in dataset:
        inputs = tokenizer("translate: " + example['oracion'], return_tensors="pt", padding="max_length", truncation=True, max_length=max_input_length)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        prepared_data.append(inputs)
    return prepared_data

# Preparar los datos de validación para la predicción
prepared_validation_data = prepare_data_for_prediction(dataset_valid, tokenizerv1)

# Lista para guardar las predicciones
predictions = []

# Generar predicciones para cada entrada en los datos de validación
for inputs in prepared_validation_data:
    with torch.no_grad():
        outputs = modelv1.generate(**inputs, max_length=max_target_length)
    decoded_output = tokenizerv1.decode(outputs[0], skip_special_tokens=True)
    predictions.append(decoded_output)

# Imprimir las predicciones
for i, prediction in enumerate(predictions):
    print(f"Prediction {i+1}: {prediction}")

Prediction 1: 
Prediction 2: 
Prediction 3: organizar 
Prediction 4: 
Prediction 5: 
Prediction 6: ordena 
Prediction 7: responder 
Prediction 8: 
Prediction 9: exponemos 
Prediction 10: yo 
Prediction 11: visión borrosa
Prediction 12: 
Prediction 13: 
Prediction 14: 
Prediction 15: berg
Prediction 16: yo 
Prediction 17: 
Prediction 18: 
Prediction 19: protegemos 
Prediction 20: 
Prediction 21: 
Prediction 22: 
Prediction 23: 
Prediction 24: 
Prediction 25: 
Prediction 26: 
Prediction 27: 
Prediction 28: 
Prediction 29: lica
Prediction 30: tráeme 
Prediction 31: 
Prediction 32: 
Prediction 33: corrige 
Prediction 34: 
Prediction 35: 
Prediction 36: 
Prediction 37: 
Prediction 38: 
Prediction 39: elaborar 
Prediction 40: 
Prediction 41: subrayo 
Prediction 42: 
Prediction 43: 
Prediction 44: 
Prediction 45: 
Prediction 46: 
Prediction 47: 
Prediction 48: 
Prediction 49: 
Prediction 50: 
Prediction 51: cartulinas 
Prediction 52: observa 
Prediction 53: 
Prediction 54: 
Prediction 55: 
Pr

In [33]:
## Pendiente de hacer pruebas

oracion_traducida = " ".join(traduccion).strip()
oracion_traducida

NameError: name 'traduccion' is not defined

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def obtener_tokens(oracion):
    resultado = []
    oracion_split = oracion.strip().split(" ")
    n = len(oracion_split)
    print(n)
    
    i = 0
    while i < n:
        elemento = oracion_split[i].strip()
        if elemento.startswith("#"):
            if elemento.endswith("#"):
                resultado.append(elemento)
                i = i + 1
            else:
                j = i + 1
                tmp = elemento
                while j < n and not oracion_split[j].endswith("#"):
                    tmp = f"{tmp} {oracion_split[j].strip()}"
                    j = j + 1
                tmp = f"{tmp} {oracion_split[j].strip()}"
                i = j + 1
                resultado.append(tmp) 
        else:
            tmp = elemento
            j = i + 1
            while j < n and not oracion_split[j].startswith("#"):
                tmp = f"{tmp} {oracion_split[j].strip()}"
                j = j + 1
            i = j
            resultado.append(tmp) 
    return resultado
    
def obtener_ruta(elemento):
  if not elemento.startswith("#"):
      return elemento
      
  valores = [elem for elem in elemento.split('#') if elem]
    
  try:
    id = int(valores[0])
  except ValueError:
    return valores[0]

  if len(valores) == 2:
    return f"imagenes_procesadas/{id}.png"

  pos = valores[2]
  return f"imagenes_procesadas/{id}_{pos}.png"

def imprimir_imagenes(rutas_imagenes):
  num_imagenes = len(rutas_imagenes)

  # Crear subplots
  fig, axes = plt.subplots(1, num_imagenes, figsize=(20, 20))
  if num_imagenes == 1:
      axes = [axes]
  for ax, ruta_imagen in zip(axes, rutas_imagenes):
      if ruta_imagen.startswith("imagenes_procesadas"):
        img = mpimg.imread(ruta_imagen)
        ax.imshow(img)
        ax.axis('off')
      else:
        ax.axis('off')  # Ocultar los ejes
        ax.text(0.5, 0.5, ruta_imagen, fontsize=12, ha='center', va='center')
  plt.tight_layout()
  plt.show()

In [ ]:
tokens = obtener_tokens(oracion_traducida) 

rutas_imagenes = []
for elemento in tokens:
  rutas_imagenes.append(obtener_ruta(elemento))

print(f"oracion original: {dataset_valid[idx].get('oracion')}")
print(f"traduccion modelo base: {dataset_valid[idx].get('traduccion')}")
print(f"traduccion modelo T5: {oracion_traducida}")
imprimir_imagenes(rutas_imagenes)


In [ ]:
dataset_valid01 = load_dataset(
    'json',
    data_files='pruebasUnitarias.json',
    split='train'
)
encoded_val_user = dataset_valid01.map(preprocess_function, batched=True)

predict01=trainer.predict(encoded_val_user,max_length=max_target_length)


In [ ]:
idx00 = 0
prediccionT5 = predict01.predictions[idx00]
prediccionT5 = np.where(prediccionT5 != -100, prediccionT5, tokenizer.pad_token_id)

traducciont5 = tokenizer.batch_decode(
        prediccionT5,
        skip_special_tokens=True
    )
print(traducciont5)
oracion_traducida_t5 = " ".join(traducciont5).strip()

oracion_traducida_t5

In [ ]:
tokenst5 = obtener_tokens(oracion_traducida_t5) 

rutas_imagenes01 = []
for elemento in tokenst5:
  rutas_imagenes01.append(obtener_ruta(elemento))


rutas_imagenes02 = []
tokensValUsuario = obtener_tokens(dataset_valid01[idx00].get('traduccion')) 

for elemento in tokensValUsuario:
  rutas_imagenes02.append(obtener_ruta(elemento))
    
print(f"oracion: {dataset_valid01[idx00].get('oracion')}")
print(f"traducción validada: {dataset_valid01[idx00].get('traduccion')}")
print(f"traducción modelo T5: {oracion_traducida_t5}")

imprimir_imagenes(rutas_imagenes02)
imprimir_imagenes(rutas_imagenes01)



In [ ]:
dataset_valid02 = load_dataset(
    'json',
    data_files='pruebasUnitarias.json',
    split='train'
)
encoded_val_user = dataset_valid02.map(preprocess_function, batched=True)

predict02=trainer.predict(encoded_val_user,max_length=max_target_length)

idx00 = 0
prediccionT501 = predict02.predictions[idx00]
prediccionT501 = np.where(prediccionT501 != -100, prediccionT501, tokenizer.pad_token_id)

traducciont501 = tokenizer.batch_decode(
        prediccionT501,
        skip_special_tokens=True
    )
print(traducciont501)
oracion_traducida_t501 = " ".join(traducciont501).strip()

tokenst501 = obtener_tokens(oracion_traducida_t501) 

rutas_imagenes03 = []
for elemento in tokenst501:
  rutas_imagenes03.append(obtener_ruta(elemento))

print(f"oracion: {dataset_valid02[idx00].get('oracion')}")
print(f"traducción modelo T5: {oracion_traducida_t501}")

imprimir_imagenes(rutas_imagenes03)